# 02 - PDF NER (Task 2: clean PDFs, Task 3: scanned PDFs)

Pipeline: PDF -> PyMuPDF -> text -> GLiNER (`gliner_small-v2.5`) -> predictions -> strict / relaxed evaluation.
The gold annotations are those of `data/plain/validation.jsonl`.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))  # makes `import ner` work from /notebooks

from ner.evaluation import evaluate_ner_relaxed, evaluate_ner_strict
from ner.inference import (
    LABEL_DESCRIPTIONS,
    get_dataset_labels,
    load_gliner_model,
    predict_dataset,
)
from ner.pdf import extract_pdf_dataset
from ner.utils import load_jsonl, save_responses_to_jsonl

VALIDATION_PATH = ROOT / "data" / "plain" / "validation.jsonl"
PDF_CLEAN_DIR = ROOT / "data" / "pdf_clean"
PDF_SCANNED_DIR = ROOT / "data" / "pdf_scanned"
RESULTS_CLEAN = ROOT / "results" / "pdf_clean" / "gliner_small_validation.jsonl"
RESULTS_SCANNED = ROOT / "results" / "pdf_scanned" / "gliner_small_validation.jsonl"
RESULTS_SCANNED_OCR = ROOT / "results" / "pdf_scanned" / "gliner_small_ocr_validation.jsonl"
THRESHOLD = 0.5

In [2]:
gold = load_jsonl(VALIDATION_PATH)
assert gold, f"{VALIDATION_PATH} is empty: copy the course data/ folder into the project first."
LABELS = get_dataset_labels(gold)
print(f"{len(gold)} gold documents | labels: {LABELS}")

assert any(PDF_CLEAN_DIR.glob("*.pdf")), f"No PDFs in {PDF_CLEAN_DIR}"
assert any(PDF_SCANNED_DIR.glob("*.pdf")), f"No PDFs in {PDF_SCANNED_DIR}"

50 gold documents | labels: ['ADDRESS', 'AMOUNT', 'DATE', 'ID_CARD_NUMBER', 'ORGANIZATION', 'PERSON_NAME', 'REGISTRATION_NUMBER']


## PyMuPDF quick start: extracted text vs. gold text

In [3]:
from ner.pdf import extract_text_from_pdf, resolve_pdf_path

doc_id = gold[0]["id"]
clean_text = extract_text_from_pdf(resolve_pdf_path(PDF_CLEAN_DIR, doc_id))
scanned_text = extract_text_from_pdf(resolve_pdf_path(PDF_SCANNED_DIR, doc_id))

print(f"gold: {len(gold[0]['text'])} chars | clean PDF: {len(clean_text)} chars | scanned PDF: {len(scanned_text)} chars")
print("--- clean ---")
print(clean_text[:400])
print("--- scanned ---")
print(repr(scanned_text[:400]))

gold: 4091 chars | clean PDF: 4089 chars | scanned PDF: 0 chars
--- clean ---
Servicio de Procedimientos Públicos de la Región Central informa de que Patricia Giménez
Domínguez, con documento de identidad 12082063W y domicilio habitual en Avenida de Aragón,
102, 2-B, 17001 Girona, presentó el 2018-04-09 una solicitud relacionada con la inscripción en un
registro administrativo sectorial. La petición se registró al sistema con el número ASS-2018-43226, y
el justificante de e
--- scanned ---
''


In [4]:
model = load_gliner_model("gliner-community/gliner_small-v2.5")

## Task 2 - Clean PDFs

In [5]:
# only some documents have a PDF: keep gold order and evaluate on those
gold_clean = [d for d in gold if any((PDF_CLEAN_DIR / f"{d['id']}{s}.pdf").exists() for s in ('', '_scanned'))]
print(f"{len(gold_clean)} of {len(gold)} documents have a clean PDF")
clean_docs = extract_pdf_dataset(PDF_CLEAN_DIR, gold_clean)
preds = predict_dataset(model, clean_docs, labels=LABELS, threshold=THRESHOLD)
save_responses_to_jsonl(preds, RESULTS_CLEAN)

strict_clean = evaluate_ner_strict(VALIDATION_PATH, RESULTS_CLEAN)
relaxed_clean = evaluate_ner_relaxed(VALIDATION_PATH, RESULTS_CLEAN)

Running GLiNER inference: 100%|██████████| 50/50 [02:35<00:00,  3.10s/it]


50 of 50 documents have a clean PDF

STRICT EVALUATION
Micro -> P: 0.53 | R: 0.60 | F1: 0.56 | TP: 1225 | FP: 1106 | FN: 825

ENTITY TYPE                    P      R     F1      TP      FP      FN
--------------------------------------------------------------------------------------------
PERSON_NAME                 0.62   0.62   0.62     308     191     192
ID_CARD_NUMBER              0.84   0.86   0.85     173      33      27
ADDRESS                     0.00   0.00   0.00       0     434     200
REGISTRATION_NUMBER         0.69   0.68   0.68     170      78      80
DATE                        0.65   0.65   0.65     261     138     139
AMOUNT                      0.79   0.77   0.78     154      41      46
ORGANIZATION                0.45   0.53   0.49     159     191     141

RELAXED EVALUATION - OCCURRENCES
Micro -> P: 0.77 | R: 0.87 | F1: 0.82 | TP: 1787 | FP: 544 | FN: 263

ENTITY TYPE                    P      R     F1      TP      FP      FN
--------------------------------------

## Task 3 - Scanned PDFs

Same pipeline on `data/pdf_scanned/`. Scanned pages may have no (or a noisy) text layer, so the number of
empty documents is printed.

In [6]:
# only some documents have a scanned version: keep gold order and evaluate on those
gold_scanned = [d for d in gold if any((PDF_SCANNED_DIR / f"{d['id']}{s}.pdf").exists() for s in ('', '_scanned'))]
print(f"{len(gold_scanned)} of {len(gold)} documents have a scanned PDF")
scanned_docs = extract_pdf_dataset(PDF_SCANNED_DIR, gold_scanned)
print("empty documents:", sum(1 for d in scanned_docs if not d["text"].strip()), "/", len(scanned_docs))

preds = predict_dataset(model, scanned_docs, labels=LABELS, threshold=THRESHOLD)
save_responses_to_jsonl(preds, RESULTS_SCANNED)

strict_scanned = evaluate_ner_strict(VALIDATION_PATH, RESULTS_SCANNED)
relaxed_scanned = evaluate_ner_relaxed(VALIDATION_PATH, RESULTS_SCANNED)

Running GLiNER inference: 100%|██████████| 10/10 [00:18<00:00,  1.82s/it]


10 of 50 documents have a scanned PDF
empty documents: 10 / 10

STRICT EVALUATION
Micro -> P: 0.00 | R: 0.00 | F1: 0.00 | TP: 0 | FP: 0 | FN: 410

ENTITY TYPE                    P      R     F1      TP      FP      FN
--------------------------------------------------------------------------------------------
PERSON_NAME                 0.00   0.00   0.00       0       0     100
ID_CARD_NUMBER              0.00   0.00   0.00       0       0      40
ADDRESS                     0.00   0.00   0.00       0       0      40
REGISTRATION_NUMBER         0.00   0.00   0.00       0       0      50
DATE                        0.00   0.00   0.00       0       0      80
AMOUNT                      0.00   0.00   0.00       0       0      40
ORGANIZATION                0.00   0.00   0.00       0       0      60

RELAXED EVALUATION - OCCURRENCES
Micro -> P: 0.00 | R: 0.00 | F1: 0.00 | TP: 0 | FP: 0 | FN: 410

ENTITY TYPE                    P      R     F1      TP      FP      FN
----------------------

### Optional - scanned PDFs with OCR

Needs Tesseract installed on the system (see the PyMuPDF OCR documentation). If it is not available the
cell reports the error and the rest of the notebook is unaffected.

In [7]:
try:
    ocr_docs = extract_pdf_dataset(PDF_SCANNED_DIR, gold_scanned, use_ocr=True)
    preds = predict_dataset(model, ocr_docs, labels=LABELS, threshold=THRESHOLD)
    save_responses_to_jsonl(preds, RESULTS_SCANNED_OCR)
    strict_ocr = evaluate_ner_strict(VALIDATION_PATH, RESULTS_SCANNED_OCR)
    relaxed_ocr = evaluate_ner_relaxed(VALIDATION_PATH, RESULTS_SCANNED_OCR)
except Exception as exc:  # Tesseract missing, etc.
    print(f"OCR skipped: {type(exc).__name__}: {exc}")

OCR skipped: RuntimeError: No OCR support: Tesseract is not installed or TESSDATA_PREFIX is not set


## Results summary (all experiments found in `results/`)

In [8]:
from ner.report import build_results_table

table, missing = build_results_table(ROOT)
print(table)
if missing:
    print("\nNot evaluated yet:", *missing, sep="\n  - ")

| Input | Model | Label descriptions | Extraction | Evaluation | P | R | F1 |
| :--- | :--- | :---: | :---: | :--- | :---: | :---: | :---: |
| Plain text | `gliner_small-v2.5` | No | — | Strict | 0.77 | 0.87 | 0.82 |
| Plain text | `gliner_small-v2.5` | No | — | Relaxed occurrences | 0.77 | 0.87 | 0.82 |
| Plain text | `gliner_small-v2.5` | No | — | Relaxed unique | 0.78 | 0.89 | 0.83 |
| Plain text | `gliner_small-v2.5` | Yes | — | Strict | 0.98 | 0.28 | 0.43 |
| Plain text | `gliner_small-v2.5` | Yes | — | Relaxed occurrences | 0.98 | 0.28 | 0.43 |
| Plain text | `gliner_small-v2.5` | Yes | — | Relaxed unique | 0.98 | 0.43 | 0.59 |
| Plain text | `gliner_large-v2.5` | No | — | Strict | 1.00 | 0.91 | 0.96 |
| Plain text | `gliner_large-v2.5` | No | — | Relaxed occurrences | 1.00 | 0.91 | 0.96 |
| Plain text | `gliner_large-v2.5` | No | — | Relaxed unique | 1.00 | 0.91 | 0.95 |
| Clean PDF | `gliner_small-v2.5` | No | PyMuPDF | Strict | 0.53 | 0.60 | 0.56 |
| Clean PDF | `gliner_small-